In [ ]:
!pip install tensorflowjs -q

import numpy as np, tensorflow as tf, tensorflowjs as tfjs, os, shutil
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

NUM_CLASSES = 5
L2 = regularizers.l2(1e-4)

def build_cnn():
    def conv_block(x, f):
        x = layers.Conv2D(f,3,padding='same',activation='relu',kernel_regularizer=L2)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(f,3,padding='same',activation='relu',kernel_regularizer=L2)(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        return x
    inp = layers.Input(shape=(224,224,3))
    x = layers.Rescaling(1./255)(inp)
    x = conv_block(x,32); x = conv_block(x,64)
    x = conv_block(x,128); x = conv_block(x,256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256,activation='relu',kernel_regularizer=L2)(x)
    x = layers.Dropout(0.6)(x)
    out = layers.Dense(NUM_CLASSES,activation='softmax')(x)
    return models.Model(inp,out,name='CNN_clean')

def build_tl():
    base = MobileNetV2(input_shape=(224,224,3),include_top=False,weights=None)
    inp = layers.Input(shape=(224,224,3))
    x = preprocess_input(inp)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128,activation='relu')(x)
    out = layers.Dense(NUM_CLASSES,activation='softmax')(x)
    return models.Model(inp,out,name='TL_clean')

def transfer_weights(src_path, clean):
    src = tf.keras.models.load_model(src_path, compile=False)
    src_layers = {}
    def collect(m):
        for l in m.layers:
            src_layers[l.name] = l
            if isinstance(l, tf.keras.Model):
                collect(l)
    collect(src)
    copied = 0
    for l in clean.layers:
        if l.name in src_layers and l.get_weights():
            try:
                l.set_weights(src_layers[l.name].get_weights()); copied += 1
            except Exception:
                pass
        elif isinstance(l, tf.keras.Model):
            for sub in l.layers:
                if sub.name in src_layers and sub.get_weights():
                    try: sub.set_weights(src_layers[sub.name].get_weights()); copied += 1
                    except Exception: pass
    return src, copied

def verify(src, clean):
    x = np.random.randint(0,256,(2,224,224,3)).astype('float32')
    a = src.predict(x, verbose=0)
    b = clean.predict(x, verbose=0)
    return float(np.max(np.abs(a-b)))

for name, builder in [('best_cnn', build_cnn), ('best_tl', build_tl)]:
    out = 'tfjs_cnn' if name=='best_cnn' else 'tfjs_tl'
    if os.path.exists(out): shutil.rmtree(out)
    clean = builder()
    src, n = transfer_weights(f'{name}.keras', clean)
    diff = verify(src, clean)
    print(f'{name}: {n} layer ter-copy, max|delta| vs asli = {diff:.6f}')
    if diff > 1e-3:
        print(f'  [PERINGATAN] selisih besar - prediksi mungkin tak cocok.')
    tfjs.converters.save_keras_model(clean, out)
    print(f'  -> {out}/', os.listdir(out))

!rm -f tfjs_models.zip
!zip -r tfjs_models.zip tfjs_cnn tfjs_tl
print('\nSelesai. Download tfjs_models.zip dari panel Files.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.
google-cloud-bigquery 3.41.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but yo

best_cnn: 18 layer ter-copy, max|delta| vs asli = 0.000000
failed to lookup keras version from the file,
    this is likely a weight only file
  -> tfjs_cnn/ ['model.json', 'group1-shard2of2.bin', 'group1-shard1of2.bin']


best_tl: 3 layer ter-copy, max|delta| vs asli = 0.000000
failed to lookup keras version from the file,
    this is likely a weight only file
  -> tfjs_tl/ ['model.json', 'group1-shard3of3.bin', 'group1-shard2of3.bin', 'group1-shard1of3.bin']
  adding: tfjs_cnn/ (stored 0%)
  adding: tfjs_cnn/model.json (deflated 93%)
  adding: tfjs_cnn/group1-shard2of2.bin (deflated 7%)
  adding: tfjs_cnn/group1-shard1of2.bin (deflated 7%)
  adding: tfjs_tl/ (stored 0%)
  adding: tfjs_tl/model.json (deflated 97%)
  adding: tfjs_tl/group1-shard3of3.bin (deflated 7%)
  adding: tfjs_tl/group1-shard2of3.bin (deflated 7%)
  adding: tfjs_tl/group1-shard1of3.bin (deflated 7%)

Selesai. Download tfjs_models.zip dari panel Files.


In [ ]:
from google.colab import files
files.download('tfjs_models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>